# Hallucination harness — quick analysis

Loads the committed `results/labels.jsonl`, prints the per-category rates,
and re-renders the bar chart inline. Re-run after `bash run.sh ...` to
see fresh numbers.

Companion to *Reliable and Responsible Foundation Models* (Yang et al., TMLR 10/2025).

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path('..').resolve()
labels = [json.loads(l) for l in open(ROOT / 'results/labels.jsonl')]
df = pd.DataFrame(labels)
summary = json.load(open(ROOT / 'results/summary.json'))

print(f"model: {df['model'].iloc[0]}")
print(f"n probes: {len(df)}")
print(f"overall hallucination rate: {summary['overall']['hallucination_rate']*100:.1f}%")
df['label'].value_counts()

In [ ]:
# Per-category breakdown
rows = []
for code, s in summary['by_category'].items():
    rows.append({**s, 'code': code, 'rate_pct': s['hallucination_rate']*100})
by_cat = pd.DataFrame(rows).sort_values('rate_pct', ascending=False)
by_cat[['code','name','n','hallucinated','correct','abstained','rate_pct']]

In [ ]:
# Re-render the plot inline
fig, ax = plt.subplots(figsize=(9, 5))
overall = summary['overall']['hallucination_rate']*100
colors = ['#1d4ed8' if r < overall + 2 else '#b91c1c' for r in by_cat['rate_pct']]
ax.bar(by_cat['name'], by_cat['rate_pct'], color=colors)
ax.axhline(overall, ls='--', color='#374151', lw=1)
ax.set_ylabel('hallucination rate (%)'); ax.set_ylim(0, 60)
ax.set_title('Hallucination rate by category')
plt.xticks(rotation=15)
plt.tight_layout(); plt.show()

In [ ]:
# Look at example hallucinations to qualitatively check the judge
df[df['label']=='H'][['id','category','question','reference','response']].head(10)

## Sanity checks tied to the survey

- **§7.1 intrinsic vs extrinsic**: extrinsic-heavy categories (`qte`, `tmp`) have higher rates than intrinsic-style (`cmn`, `pop`). Consistent.
- **§9 distribution shift**: temporal claims hallucinate more — model trained on stale web data.
- **§7.4 mitigation focus**: top two failing categories should drive RAG / retrieval grounding.